In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import joblib

df = pd.read_csv("../data/raw/cold_storage_raw.csv")

df.head()

df = df.drop_duplicates()

df['timestamp'] = pd.to_datetime(df['timestamp'])

df['hour'] = df['timestamp'].dt.hour
df['dayofweek'] = df['timestamp'].dt.dayofweek

df.to_csv("../data/processed/telemetry_clean.csv", index=False)

feature_cols = [
    'Temperature',
    'Humidity',
    'CO2',
    'DoorOpen',
    'PowerConsumption',
    'CompressorVibration',
    'hour',
    'dayofweek'
]

train_size = int(len(df) * 0.75)

train_df = df[:train_size]
test_df = df[train_size:]

X_train = train_df[feature_cols]
y_train = train_df['CoolingRisk']

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

X_test = test_df[feature_cols]
y_test = test_df['CoolingRisk']

preds = model.predict(X_test)

accuracy = accuracy_score(y_test, preds)

print(accuracy)

joblib.dump(model, "../models/cold_storage_model.joblib")
import json

metrics = {
    "accuracy": float(accuracy)
}

with open("../outputs/metrics.json", "w") as f:
    json.dump(metrics, f)
test_df['prediction'] = preds

test_df['decision'] = test_df['prediction'].apply(
    lambda x: "CHECK_COOLING" if x == 1 else "NORMAL"
)

test_df.to_csv("../outputs/decision_log.csv", index=False)

df.to_csv("../data/processed/feature_dataset.csv", index=False)

test_df.to_csv("../outputs/predictions.csv", index=False)

1.0
